In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


def metricas_classificacao(y_true, y_pred):
    report = classification_report(
        y_true,
        y_pred,
        output_dict=True,
        zero_division=0
    )
    tabela = pd.DataFrame(report).transpose()
    tabela = tabela.drop(index='accuracy', errors='ignore')
    tabela = tabela[['precision', 'recall', 'f1-score', 'support']]
    tabela['support'] = tabela['support'].astype(int)
    return tabela

# 1. Carregar os dados
df_text = pd.read_csv('../data/Mental Health Disorder Detection Dataset.csv')

coluna_texto = 'body'
coluna_alvo = 'category'

df_text = df_text.dropna(subset=[coluna_texto, coluna_alvo])

X_text = df_text[coluna_texto]
y_text = df_text[coluna_alvo]

print('Distribuicao das categorias:')
print(y_text.value_counts().sort_index())

# 2. Divisao treino/teste estratificada
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_text, y_text, test_size=0.2, random_state=42, stratify=y_text
)

# 3. TF-IDF
print('Vetorizando textos...')
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train_t)
X_test_vec = vectorizer.transform(X_test_t)

# 4. Modelo com pesos balanceados
modelo_nlp = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
modelo_nlp.fit(X_train_vec, y_train_t)

# 5. Resultados
y_pred_t = modelo_nlp.predict(X_test_vec)

print('\n=== RELATORIO DE CLASSIFICACAO NLP ===')
print(metricas_classificacao(y_test_t, y_pred_t).to_string(float_format=lambda value: f'{value:.4f}'))

# 6. Matriz de Confusao
plt.figure(figsize=(10, 7))
sns.heatmap(confusion_matrix(y_test_t, y_pred_t), annot=True, fmt='d', cmap='Oranges')
plt.title('Matriz de Confusao - NLP')
plt.show()
